In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import warnings

BASE_PATH = "/Users/dominicranelli/Desktop/GQPData"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 10

# Device selection
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

/opt/anaconda3/envs/modernbert_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


In [3]:

class SequentialNAICSDataset(Dataset):
    def __init__(self, texts, given_codes, labels, tokenizer, max_len):
        self.texts = texts
        self.given_codes = given_codes
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        encoding = self.tokenizer(
            str(self.texts[item]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'given_code': torch.tensor(self.given_codes[item], dtype=torch.long),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

In [4]:
class SequentialNAICSModel(nn.Module):
    def __init__(self, n_classes, n_given_levels):
        super(SequentialNAICSModel, self).__init__()
        self.bert = AutoModel.from_pretrained("answerdotai/ModernBERT-base")
        
        self.use_context = n_given_levels > 0
        if self.use_context:
            self.context_embedding = nn.Embedding(n_given_levels, 64)
            input_dim = 768 + 64
        else:
            input_dim = 768

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, n_classes)
        )

    def forward(self, input_ids, attention_mask, given_code):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]
        
        if self.use_context:
            ctx = self.context_embedding(given_code)
            combined = torch.cat((pooled, ctx), dim=1)
        else:
            combined = pooled
            
        return self.classifier(combined)

In [5]:
def process_level_data(level_num):
    file_path = f"d{level_num}_train.csv"
    df = pd.read_csv(file_path)
    
    df['text_input'] = (
        df['Business_Name'].fillna('') + " [SEP] " + 
        df['FULL_ADDRESS'].fillna('')
    ).str.lower()


    le_target = LabelEncoder()
    df['y'] = le_target.fit_transform(df['predictor'].astype(str))
    
    # Given context
    num_parent_codes = 0
    if 'given' in df.columns:
        le_given = LabelEncoder()
        df['parent_idx'] = le_given.fit_transform(df['given'].astype(str))
        num_parent_codes = len(le_given.classes_)
    else:
        df['parent_idx'] = 0
        
    return df, len(le_target.classes_), num_parent_codes

In [6]:
def main(level_to_train):
    df, n_classes, n_parents = process_level_data(level_to_train)
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

    train_df, val_df = train_test_split(df, test_size=0.1, stratify=df['y'], random_state=42)
    
    train_ds = SequentialNAICSDataset(train_df['text_input'].values, train_df['parent_idx'].values, 
                                      train_df['y'].values, tokenizer, MAX_LEN)
    val_ds = SequentialNAICSDataset(val_df['text_input'].values, val_df['parent_idx'].values, 
                                    val_df['y'].values, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    model = SequentialNAICSModel(n_classes, n_parents).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    loss_fn = nn.CrossEntropyLoss()

    print(f"--- Training: {level_to_train} ---")
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            logits = model(batch['input_ids'].to(DEVICE), 
                           batch['attention_mask'].to(DEVICE), 
                           batch['given_code'].to(DEVICE))
            loss = loss_fn(logits, batch['labels'].to(DEVICE))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Eval
        model.eval()
        all_preds = []
        all_actuals = []
        with torch.no_grad():
            for batch in val_loader:
                logits = model(batch['input_ids'].to(DEVICE), 
                               batch['attention_mask'].to(DEVICE), 
                               batch['given_code'].to(DEVICE))
                all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                all_actuals.extend(batch['labels'].cpu().numpy())
        
        # Metrics
        acc = accuracy_score(all_actuals, all_preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_actuals, all_preds, average='weighted', zero_division=0
        )
        
        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"Loss: {total_loss/len(train_loader):.4f}")
        print(f"Accuracy:  {acc:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1-Score:  {f1:.4f}")
        print("-" * 30)

if __name__ == "__main__":
    main(level_to_train=1)